In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\dhima\Downloads\student_performance_project.csv")
print(df.shape)
print(df.head())

(61, 7)
   StudentID          Name Gender       City  Math  Science  English
0          1  Aditi Sharma      F  Bengaluru  33.0     33.0     74.0
1          2   Rohan Verma      F     Mumbai  78.0     97.0     60.0
2          3    Meera Nair      F      Delhi  90.0     81.0     70.0
3          4   Karan Mehta      F      Delhi  45.0     41.0     70.0
4          5     Sneha Rao      M      Delhi  91.0     80.0     51.0


In [2]:
print(df.isnull().sum())
print('Duplicate rows: ',df.duplicated().sum())

StudentID    0
Name         0
Gender       0
City         0
Math         4
Science      3
English      3
dtype: int64
Duplicate rows:  1


In [3]:
for subject in['Math' , 'Science' , 'English']:
    df[subject] = df[subject].fillna(df[subject].mean()).round(1)
print(df.isnull().sum())

StudentID    0
Name         0
Gender       0
City         0
Math         0
Science      0
English      0
dtype: int64


In [4]:
df = df.drop_duplicates(subset=['Name','City'], keep='first')
print(df.shape)

(60, 7)


In [6]:
top_math = df[df['Math'] > 80]
print('Student above 80 in Math: ',len(top_math))


delhi_science = df[(df['City'] == 'Delhi') & (df['Science'] >= 70)]
print(delhi_science[['StudentID' , 'Name' , 'City' , 'Science']])

Student above 80 in Math:  13
   StudentID        Name   City  Science
2          3  Meera Nair  Delhi     81.0
4          5   Sneha Rao  Delhi     80.0


In [7]:
subject_avg = df[['Math' , 'Science' , 'English']].mean().round(2)
print(subject_avg)

Math       62.47
Science    63.11
English    63.29
dtype: float64


In [8]:
df['Total'] = df[['Math' , 'Science' , 'English']].sum(axis = 1).round(1)
df['Average'] = df[['Math' , 'Science' , 'English']].mean(axis=1).round(2)
df['Rank'] = df['Total'].rank(ascending=False , method='min').astype(int)
df = df.sort_values('Rank')
print(df[['Rank' , 'StudentID' , 'Name' , 'Total','Average']].head(6))


    Rank  StudentID             Name  Total  Average
51     1         52    Lavanya Menon  287.0    95.67
47     2         48     Hitesh Goyal  279.0    93.00
37     3         38    Sameer Ansari  260.0    86.67
2      4          3       Meera Nair  241.0    80.33
52     5         53  Mohit Ahluwalia  238.5    79.50
1      6          2      Rohan Verma  235.0    78.33


In [9]:
PASS_MARK = 40
df['Result'] = np.where(
    (df['Math'] >= PASS_MARK) & (df['Science'] >= PASS_MARK) & (df['English'] >= PASS_MARK), 'Pass' , 'Fail'
)
print(df['Result'].value_counts())

Result
Pass    40
Fail    20
Name: count, dtype: int64


In [12]:
gender_report = df.groupby('Gender')['Average'].mean().round(2)
city_report = df.groupby('City')['Average'].mean().round(2)
print(gender_report)
print(city_report)

Gender
F    63.89
M    61.74
Name: Average, dtype: float64
City
Bengaluru    54.37
Chennai      67.59
Delhi        66.35
Kolkata      58.30
Mumbai       63.84
Pune         66.32
Name: Average, dtype: float64


In [13]:
summary = df.groupby(['City','Gender']).agg(
    students=('StudentID','count'),
    avg_total=('Total','mean'),
    pass_rate = ('Result' , lambda x : (x == 'Pass').mean() * 100)
).round(1).reset_index()
print(summary)

         City Gender  students  avg_total  pass_rate
0   Bengaluru      F         5      168.2       60.0
1   Bengaluru      M         3      154.7        0.0
2     Chennai      F         6      205.2       66.7
3     Chennai      M         5      199.9       80.0
4       Delhi      F         5      197.5      100.0
5       Delhi      M         3      201.7      100.0
6     Kolkata      F         3      159.3        0.0
7     Kolkata      M         8      180.8       50.0
8      Mumbai      F        10      189.6       80.0
9      Mumbai      M         3      198.0      100.0
10       Pune      F         5      216.7       80.0
11       Pune      M         4      176.8       50.0


In [14]:
topper_list = df.sort_values('Total' , ascending=False)[['Rank' , 'Name' , 'City' , 'Total' , 'Average' , 'Result']]
print(topper_list.head(5))

    Rank             Name     City  Total  Average Result
51     1    Lavanya Menon     Pune  287.0    95.67   Pass
47     2     Hitesh Goyal     Pune  279.0    93.00   Pass
37     3    Sameer Ansari  Chennai  260.0    86.67   Pass
2      4       Meera Nair    Delhi  241.0    80.33   Pass
52     5  Mohit Ahluwalia  Chennai  238.5    79.50   Pass


In [19]:
# df.to_csv(r"C:\Users\dhima\Downloads\student_performance_project.csv",index=False)
# df.to_excel(r"C:\Users\dhima\Downloads\students_raw.xlsx" , sheet_name='Report',index=False)
# print('Report exposed successfully:', df.shape)